# Aligned stability sweep versus batch size

This notebook varies batch size while matching the problem, optimizer, learning-rate grid, run length, and seeds used by `momentum_gap.ipynb`.

It scans the stability boundary of:

- `SGD`
- `Polyak`
- `Nesterov`

For each `(algorithm, B, eta)`, the experiment runs `N_SEEDS=5` independent seeds.
For each seed and batch size, it finds the largest stable learning rate; the plotted
boundary is the average of these per-seed largest stable learning rates.

The fixed damping $1-\rho=2^{-8}$ is one point in the companion momentum-gap grid, while that experiment's fixed $B=16$ is one point in this batch-size grid.
This cross-design makes the fitted constants directly comparable.

For momentum methods, the damping is fixed at

\[
1-\rho = 2^{-8}.
\]

The three dashed prediction lines use the reference scalings

\[
\eta_{\mathrm{SGD}}^{\mathrm{crit}}\asymp 1,\qquad
\eta_{\mathrm{Polyak}}^{\mathrm{crit}}\asymp \min\{1,B(1-\rho)\},\qquad
\eta_{\mathrm{Nesterov}}^{\mathrm{crit}}\asymp \min\{1,B^\beta(1-\rho)\}.
\]


In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

# ----------------------------- device -----------------------------
if torch.cuda.is_available():
    DEVICE, DTYPE = torch.device('cuda'), torch.float64
    torch.backends.cuda.matmul.allow_tf32 = False
elif torch.backends.mps.is_available():
    DEVICE, DTYPE = torch.device('mps'), torch.float32
else:
    DEVICE, DTYPE = torch.device('cpu'), torch.float64
print(f'device={DEVICE}, dtype={DTYPE}')

# ----------------------------- paper-scale experiment settings -----------------------------
BETA = 1.5
S_VALUE = 0.3
SIGMA = 0.0
DIM = 4096

# ----------------------------- stability experiment parameters -----------------------------
ONE_MINUS_RHO = 2.0 ** -8
RHO = 1.0 - ONE_MINUS_RHO

K_STEPS = 2000
WINDOW = 20
STABLE_THRESHOLD = 1
DIVERGENCE_LOSS = 1e12

B_EXPONENTS = np.arange(2, 11)
B_LIST = np.rint(2 ** B_EXPONENTS).astype(int)

ETA_MIN = 2 ** -10
ETA_MAX = 8
ETA_POINTS = 50
ETA_GRID = np.geomspace(ETA_MIN, ETA_MAX, ETA_POINTS)

ALGOS = {
    'SGD': 'sgd',
    'Polyak': 'polyak',
    'Nesterov': 'nesterov',
}
ALGO_SEED_OFFSET = {name: k for k, name in enumerate(ALGOS)}

SEED = 20260822
SEED_SCHEME = 'aligned-v2: algorithm, seed, log2(B), -2log2(1-rho), eta-index'
N_SEEDS = 5
CHUNK = 2048
LOAD_CACHE = True
SAVE_CACHE = True
RESULT_DIR = Path('stability/results/generated/aligned/batch_size')
RESULT_DIR.mkdir(parents=True, exist_ok=True)

def aligned_trial_seed(algorithm_index, seed_index, batch_size, one_minus_rho, eta_index):
    """Shared seed map; identical settings give identical samples in both notebooks."""
    batch_code = int(round(np.log2(float(batch_size))))
    gap_half_exponent_code = int(round(-2.0 * np.log2(float(one_minus_rho))))
    return int(
        SEED
        + algorithm_index * 100_000_000
        + seed_index * 10_000_000
        + batch_code * 100_000
        + gap_half_exponent_code * 100
        + eta_index
    )

print(f'beta={BETA:g}, sigma={SIGMA:g}, dim={DIM}, s={S_VALUE:g}')
print(f'1-rho={ONE_MINUS_RHO:.2e}, rho={RHO:.12f} (momentum methods only)')
print(f'B grid: 2^{B_EXPONENTS[0]} ... 2^{B_EXPONENTS[-1]} ({len(B_LIST)} values)')
print(f'eta grid: {ETA_MIN:.1e} ... {ETA_MAX:.1e} ({len(ETA_GRID)} values)')
print(f'stability: final {WINDOW}-step moving-average loss < {STABLE_THRESHOLD:g} after {K_STEPS} steps')
print(f'seeds per (B, eta): {N_SEEDS}')
print(f'minimum resolved damping time: K(1-rho)={K_STEPS * ONE_MINUS_RHO:.3f}')


## Linear-regression problem and stochastic gradient

We work in the eigenbasis of the diagonal covariance matrix. The exact population
excess loss is

\[
\mathcal E(\theta)=\frac12\sum_j \lambda_j(\theta_j-\theta^*_j)^2.
\]

The stochastic gradient is computed in chunks so large batches do not require
materializing very large tensors at once.


In [ ]:
@torch.no_grad()
def build_problem(dim=DIM, beta=BETA, s=S_VALUE, device=DEVICE, dtype=DTYPE):
    """Return (lam, theta_star) for the diagonal linear-regression problem."""
    j = torch.arange(1, dim + 1, device=device, dtype=dtype)
    lam = j.pow(-beta)
    var_star = j.pow(-(1.0 + (s - 1.0) * beta))
    theta_star = torch.sqrt(var_star)
    return lam, theta_star


@torch.no_grad()
def excess_loss(delta, lam):
    """Exact population excess loss E = 1/2 * sum_j lam_j * delta_j^2."""
    return 0.5 * torch.sum(lam * delta * delta)


@torch.no_grad()
def grad_delta_chunked(delta, B, lam, sigma=SIGMA, generator=None, chunk=CHUNK):
    """Stochastic minibatch gradient without materializing a full (B, DIM) batch."""
    d = lam.numel()
    sqrt_lam = torch.sqrt(lam)
    grad = torch.zeros_like(delta)
    remaining = int(B)
    while remaining > 0:
        b = min(chunk, remaining)
        X = torch.randn((b, d), device=delta.device, dtype=delta.dtype, generator=generator)
        X.mul_(sqrt_lam)
        resid = X.mv(delta)
        if sigma != 0:
            eps = torch.randn((b,), device=delta.device, dtype=delta.dtype, generator=generator)
            resid.sub_(eps.mul_(sigma))
        grad.add_(X.t().mv(resid))
        remaining -= b
    grad.div_(float(B))
    return grad


lam, theta_star = build_problem()
E0 = float(excess_loss(-theta_star, lam).item())
print(f'lambda range [{float(lam[-1].item()):.2e}, {float(lam[0].item()):.2e}], initial excess loss E(0)={E0:.4f}')


## Stability test

For each `(algorithm, B, eta)`, run exactly `K_STEPS` updates. The final moving
average is computed over the last `WINDOW` exact population excess losses. A run is
stable if this final moving average is finite and below `STABLE_THRESHOLD`.

SGD always uses `rho=0`. Polyak and Nesterov use the configured `RHO`.


In [ ]:
@torch.no_grad()
def run_stability(algo, B, eta, rho=RHO, K=K_STEPS, window=WINDOW,
                  threshold=STABLE_THRESHOLD, divergence_loss=DIVERGENCE_LOSS,
                  seed=SEED):
    """Return a dict with stable flag, final moving average, and final loss."""
    generator = torch.Generator(device=lam.device)
    generator.manual_seed(int(seed))
    rho_eff = 0.0 if algo == 'sgd' else rho

    delta = -theta_star.clone()
    delta_prev = delta.clone()
    tail = []

    loss = float(excess_loss(delta, lam).item())
    tail.append(loss)

    for t in range(1, K + 1):
        momentum = delta - delta_prev
        eval_delta = delta + rho_eff * momentum if algo == 'nesterov' else delta
        g = grad_delta_chunked(eval_delta, B, lam, generator=generator)
        delta_new = delta - eta * g + rho_eff * momentum
        delta_prev, delta = delta, delta_new

        loss = float(excess_loss(delta, lam).item())
        if (not np.isfinite(loss)) or loss > divergence_loss:
            return {
                'stable': False,
                'final_ma': np.inf,
                'final_loss': loss if np.isfinite(loss) else np.inf,
                'steps': t,
                'reason': 'diverged',
            }
        tail.append(loss)
        if len(tail) > window:
            tail.pop(0)

    final_ma = float(np.mean(tail))
    return {
        'stable': bool(np.isfinite(final_ma) and final_ma < threshold),
        'final_ma': final_ma,
        'final_loss': float(loss),
        'steps': K,
        'reason': 'finished',
    }


quick = run_stability('sgd', B=1, eta=ETA_GRID[0], K=10, window=10)
quick


## Run the stability grid

Aligned results are cached under `stability/results/generated/aligned/batch_size/`.
The cache filename records the device, problem exponents, batch grid, damping, run
length, threshold, and seed so an older stability scan cannot be loaded silently.

For each `(algorithm, B, eta)`, this cell runs `N_SEEDS=5` independent seeds.  The
plotted stability boundary is not based on a single run: for each seed we find the
largest stable learning rate, then average those five largest stable learning rates.


In [ ]:
def cache_path():
    device_tag = DEVICE.type
    dtype_tag = str(DTYPE).split('.')[-1]
    tag = (
        f'stability_phase_gpu_3alg_seed{N_SEEDS}_{device_tag}_{dtype_tag}_beta{BETA:g}_s{S_VALUE:g}_dim{DIM}_'
        f'B{B_EXPONENTS[0]}-{B_EXPONENTS[-1]}_eta{ETA_POINTS}_'
        f'omrho{ONE_MINUS_RHO:.0e}_K{K_STEPS}_w{WINDOW}_thr{STABLE_THRESHOLD:g}_seed{SEED}_seedv2'
    ).replace('.', '_').replace('-', 'm')
    return RESULT_DIR / f'{tag}.npz'


def max_stable_eta_per_seed(stable_cube):
    """Largest stable eta for each seed and B; NaN if no eta is stable."""
    out = np.full((stable_cube.shape[0], stable_cube.shape[2]), np.nan)
    for seed_idx in range(stable_cube.shape[0]):
        for j in range(stable_cube.shape[2]):
            idx = np.where(stable_cube[seed_idx, :, j])[0]
            if len(idx):
                out[seed_idx, j] = ETA_GRID[idx.max()]
    return out


def mean_max_stable_eta(stable_cube):
    """Average per-seed largest stable eta for each B."""
    per_seed = max_stable_eta_per_seed(stable_cube)
    out = np.full(per_seed.shape[1], np.nan)
    for j in range(per_seed.shape[1]):
        vals = per_seed[:, j]
        vals = vals[np.isfinite(vals)]
        if len(vals):
            out[j] = float(np.mean(vals))
    return out


def summarize_min_loss_by_B(final_ma, final_loss):
    """For each algorithm and B, record the lowest loss over all seeds and eta values."""
    min_final_ma = {}
    min_final_ma_eta = {}
    min_final_loss = {}
    min_final_loss_eta = {}

    for name in ALGOS:
        min_final_ma[name] = np.full(len(B_LIST), np.nan)
        min_final_ma_eta[name] = np.full(len(B_LIST), np.nan)
        min_final_loss[name] = np.full(len(B_LIST), np.nan)
        min_final_loss_eta[name] = np.full(len(B_LIST), np.nan)

        for j in range(len(B_LIST)):
            ma_slice = np.asarray(final_ma[name][:, :, j], dtype=float)
            loss_slice = np.asarray(final_loss[name][:, :, j], dtype=float)

            ma_finite = np.isfinite(ma_slice)
            if np.any(ma_finite):
                flat_idx = int(np.argmin(np.where(ma_finite, ma_slice, np.inf)))
                seed_idx, eta_idx = np.unravel_index(flat_idx, ma_slice.shape)
                min_final_ma[name][j] = ma_slice[seed_idx, eta_idx]
                min_final_ma_eta[name][j] = ETA_GRID[eta_idx]

            loss_finite = np.isfinite(loss_slice)
            if np.any(loss_finite):
                flat_idx = int(np.argmin(np.where(loss_finite, loss_slice, np.inf)))
                seed_idx, eta_idx = np.unravel_index(flat_idx, loss_slice.shape)
                min_final_loss[name][j] = loss_slice[seed_idx, eta_idx]
                min_final_loss_eta[name][j] = ETA_GRID[eta_idx]

    return min_final_ma, min_final_ma_eta, min_final_loss, min_final_loss_eta


path = cache_path()
if LOAD_CACHE and path.exists():
    data = np.load(path, allow_pickle=True)
    STABLE = data['stable'].item()
    FINAL_MA = data['final_ma'].item()
    FINAL_LOSS = data['final_loss'].item()
    STEPS_DONE = data['steps_done'].item()
    if 'min_final_ma' in data.files:
        MIN_FINAL_MA = data['min_final_ma'].item()
        MIN_FINAL_MA_ETA = data['min_final_ma_eta'].item()
        MIN_FINAL_LOSS = data['min_final_loss'].item()
        MIN_FINAL_LOSS_ETA = data['min_final_loss_eta'].item()
    else:
        MIN_FINAL_MA, MIN_FINAL_MA_ETA, MIN_FINAL_LOSS, MIN_FINAL_LOSS_ETA = summarize_min_loss_by_B(FINAL_MA, FINAL_LOSS)
    print(f'loaded cached stability grid from {path}')
else:
    STABLE = {name: np.zeros((N_SEEDS, len(ETA_GRID), len(B_LIST)), dtype=bool) for name in ALGOS}
    FINAL_MA = {name: np.full((N_SEEDS, len(ETA_GRID), len(B_LIST)), np.nan) for name in ALGOS}
    FINAL_LOSS = {name: np.full((N_SEEDS, len(ETA_GRID), len(B_LIST)), np.nan) for name in ALGOS}
    STEPS_DONE = {name: np.zeros((N_SEEDS, len(ETA_GRID), len(B_LIST)), dtype=int) for name in ALGOS}

    for name, algo in ALGOS.items():
        print(f'=== {name} ===')
        for j, B in enumerate(B_LIST):
            print(f'B=2^{int(np.log2(B)):<2d} ({B:6d})')
            for i, eta in enumerate(ETA_GRID):
                for seed_idx in range(N_SEEDS):
                    run_seed = aligned_trial_seed(
                        ALGO_SEED_OFFSET[name], seed_idx, int(B), ONE_MINUS_RHO, i
                    )
                    out = run_stability(algo, int(B), float(eta), seed=run_seed)
                    STABLE[name][seed_idx, i, j] = out['stable']
                    FINAL_MA[name][seed_idx, i, j] = out['final_ma']
                    FINAL_LOSS[name][seed_idx, i, j] = out['final_loss']
                    STEPS_DONE[name][seed_idx, i, j] = out['steps']

            n_stable = int(STABLE[name][:, :, j].sum())
            boundary_seeds = max_stable_eta_per_seed(STABLE[name])[:, j]
            boundary_mean = float(np.nanmean(boundary_seeds)) if np.any(np.isfinite(boundary_seeds)) else np.nan
            finite = np.isfinite(FINAL_MA[name][:, :, j])
            if np.any(finite):
                flat_idx = int(np.argmin(np.where(finite, FINAL_MA[name][:, :, j], np.inf)))
                seed_best, eta_best = np.unravel_index(flat_idx, FINAL_MA[name][:, :, j].shape)
                print(f'  stable runs: {n_stable}/{N_SEEDS * len(ETA_GRID)}; '
                      f'mean max stable eta={boundary_mean:.3e}; '
                      f'min final-MA loss={FINAL_MA[name][seed_best, eta_best, j]:.3e} at eta={ETA_GRID[eta_best]:.3e}')
            else:
                print(f'  stable runs: {n_stable}/{N_SEEDS * len(ETA_GRID)}; mean max stable eta=nan; min final-MA loss=nan')

    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()

    MIN_FINAL_MA, MIN_FINAL_MA_ETA, MIN_FINAL_LOSS, MIN_FINAL_LOSS_ETA = summarize_min_loss_by_B(FINAL_MA, FINAL_LOSS)

    if SAVE_CACHE:
        np.savez_compressed(
            path,
            stable=STABLE,
            final_ma=FINAL_MA,
            final_loss=FINAL_LOSS,
            steps_done=STEPS_DONE,
            min_final_ma=MIN_FINAL_MA,
            min_final_ma_eta=MIN_FINAL_MA_ETA,
            min_final_loss=MIN_FINAL_LOSS,
            min_final_loss_eta=MIN_FINAL_LOSS_ETA,
            B_LIST=B_LIST,
            ETA_GRID=ETA_GRID,
            N_SEEDS=N_SEEDS,
        )
        print(f'saved stability grid to {path}')


## Plot the batch-size stability boundary

This plot puts SGD, Polyak, and Nesterov in the same axes. Points are the average,
over `N_SEEDS=5`, of each seed's largest stable learning rate at each batch size.
Both momentum methods use the same continuous one-kink model as the companion
momentum-gap notebook. The normalized variable is chosen before fitting, so the
fitted coefficient has the same definition in both sweeps:

\[
\eta_{\mathrm{SGD}}^{\mathrm{crit}}\asymp 1,\qquad
\eta_{\mathrm{Polyak}}^{\mathrm{crit}}\asymp \min\{1,B(1-\rho)\},\qquad
\eta_{\mathrm{Nesterov}}^{\mathrm{crit}}\asymp \min\{1,B^\beta(1-\rho)\}.
\]

The styling uses a white background, no grid or region fills, green/blue/orange
markers, and prediction-only legend entries. The summary JSON records the normalized
coefficient, ceiling, and kink location for direct cross-experiment comparison.


In [ ]:
def fit_linear_one_kink(values, scale, min_points_per_side=2, grid_size=4001):
    """Fit eta_crit = min(level, c * scale) with unit log-log slope."""
    finite = np.isfinite(values) & (values > 0) & np.isfinite(scale) & (scale > 0)
    x, y = np.log(scale[finite]), np.log(values[finite])
    order = np.argsort(x)
    x, y = x[order], y[order]
    if x.size < 2 * min_points_per_side:
        raise ValueError('not enough finite points to fit both sides of one kink')

    candidates = np.linspace(
        x[min_points_per_side - 1],
        x[-min_points_per_side],
        grid_size,
    )
    best = None
    for log_breakpoint in candidates:
        hinge = np.minimum(x - log_breakpoint, 0.0)
        log_level = float(np.mean(y - hinge))
        residual = y - (log_level + hinge)
        sse = float(residual @ residual)
        if best is None or sse < best['sse_log']:
            level = float(np.exp(log_level))
            breakpoint = float(np.exp(log_breakpoint))
            best = {
                'coefficient': level / breakpoint,
                'level': level,
                'breakpoint_scale': breakpoint,
                'slope': 1.0,
                'sse_log': sse,
            }
    return best


BOUNDARIES = {
    name: mean_max_stable_eta(STABLE[name])
    for name in ALGOS
}

batch_float = B_LIST.astype(float)
NORMALIZED_SCALE = {
    'Polyak': batch_float * ONE_MINUS_RHO,
    'Nesterov': batch_float**BETA * ONE_MINUS_RHO,
}

FITS = {
    'sgd': {
        'level': float(np.exp(np.nanmean(np.log(BOUNDARIES['SGD'])))),
        'slope': 0.0,
    },
    'polyak': fit_linear_one_kink(
        BOUNDARIES['Polyak'],
        NORMALIZED_SCALE['Polyak'],
    ),
    'nesterov': fit_linear_one_kink(
        BOUNDARIES['Nesterov'],
        NORMALIZED_SCALE['Nesterov'],
    ),
}
FITS['polyak']['coefficient_vs_B'] = (
    FITS['polyak']['coefficient'] * ONE_MINUS_RHO
)
FITS['nesterov']['coefficient_vs_B_beta'] = (
    FITS['nesterov']['coefficient'] * ONE_MINUS_RHO
)

COLORS = {
    'SGD': 'tab:green',
    'Polyak': 'tab:blue',
    'Nesterov': 'tab:orange',
}
MARKERS = {'SGD': '^', 'Polyak': 'o', 'Nesterov': 's'}

plt.rcParams.update({
    'font.family': 'sans-serif',
    'axes.labelsize': 16,
    'axes.titlesize': 20,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 10,
})
fig, ax = plt.subplots(figsize=(5.0, 4.0))

for name in ALGOS:
    values = BOUNDARIES[name]
    mask = np.isfinite(values) & (values > 0)
    ax.plot(
        B_LIST[mask],
        values[mask],
        linestyle='None',
        marker=MARKERS[name],
        markersize=6,
        color=COLORS[name],
        markerfacecolor=COLORS[name],
        markeredgecolor=COLORS[name],
        markeredgewidth=1.1,
        label='_nolegend_',
        zorder=3,
    )

dense_B = np.geomspace(B_LIST.min(), B_LIST.max(), 500)
ax.plot(
    dense_B,
    np.full_like(dense_B, FITS['sgd']['level']),
    '--',
    color=COLORS['SGD'],
    linewidth=1.8,
    label='SGD Prediction',
    zorder=2,
)
for name, key, power in (
    ('Polyak', 'polyak', 1.0),
    ('Nesterov', 'nesterov', BETA),
):
    scale = dense_B**power * ONE_MINUS_RHO
    prediction = np.minimum(
        FITS[key]['level'],
        FITS[key]['coefficient'] * scale,
    )
    ax.plot(
        dense_B,
        prediction,
        '--',
        color=COLORS[name],
        linewidth=1.8,
        label=f'{name} Prediction',
        zorder=2,
    )

ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlim(B_LIST.min(), B_LIST.max())
ax.set_xticks(2.0**B_EXPONENTS)
ax.set_xticklabels([rf'$2^{{{exponent}}}$' for exponent in B_EXPONENTS])
ax.set_ylim(float(ETA_GRID.min()), float(ETA_GRID.max()))
ax.set_xlabel(r'Batch Size  $B$')
ax.set_ylabel(r'Stability Boundary  $\eta$')
ax.set_title('Critical learning rates')
ax.grid(False)
ax.legend(loc='lower right', handlelength=2.8)

PLOT_STEM = RESULT_DIR / 'aligned_batch_size'
fig.tight_layout()
fig.savefig(PLOT_STEM.with_suffix('.png'), dpi=180, bbox_inches='tight')
fig.savefig(PLOT_STEM.with_suffix('.pdf'), bbox_inches='tight')
plt.show()

cross_batch_index = int(np.flatnonzero(B_LIST == 16)[0])
summary = {
    'config': {
        'beta': BETA,
        's': S_VALUE,
        'sigma': SIGMA,
        'dim': DIM,
        'fixed_one_minus_rho': ONE_MINUS_RHO,
        'batch_sizes': B_LIST.tolist(),
        'eta_grid': ETA_GRID.tolist(),
        'steps': K_STEPS,
        'window': WINDOW,
        'n_seeds': N_SEEDS,
        'seed': SEED,
        'seed_scheme': SEED_SCHEME,
    },
    'fit_model': 'eta_crit = min(level, coefficient * normalized_scale)',
    'normalized_scale': {
        'Polyak': 'B * (1-rho)',
        'Nesterov': 'B**beta * (1-rho)',
    },
    'fits': FITS,
    'boundary': {name: BOUNDARIES[name].tolist() for name in ALGOS},
    'cross_point': {
        'batch_size': 16,
        'one_minus_rho': ONE_MINUS_RHO,
        'boundary': {
            name: float(BOUNDARIES[name][cross_batch_index]) for name in ALGOS
        },
    },
}
with (RESULT_DIR / 'aligned_batch_size_summary.json').open('w') as f:
    json.dump(summary, f, indent=2)

print(f"Polyak normalized coefficient c={FITS['polyak']['coefficient']:.6g}")
print(f"Nesterov normalized coefficient c={FITS['nesterov']['coefficient']:.6g}")
print(f"Polyak raw coefficient versus B={FITS['polyak']['coefficient_vs_B']:.6g}")
print(f"Nesterov raw coefficient versus B**beta={FITS['nesterov']['coefficient_vs_B_beta']:.6g}")


## Boundary and minimum-loss table

This table prints both the largest stable `eta` and the lowest final moving-average
loss for each batch size.


In [ ]:
for name in ALGOS:
    per_seed_boundary = max_stable_eta_per_seed(STABLE[name])
    boundary = mean_max_stable_eta(STABLE[name])
    print(f'\n{name}: mean max stable eta over {N_SEEDS} seeds and minimum loss by B')
    for j, (B, eta_mean) in enumerate(zip(B_LIST, boundary)):
        eta_text = f'{eta_mean:.3e}' if np.isfinite(eta_mean) else 'nan'
        seed_vals = per_seed_boundary[:, j]
        seed_vals = seed_vals[np.isfinite(seed_vals)]
        eta_std_text = f'{np.std(seed_vals):.3e}' if len(seed_vals) else 'nan'
        min_ma = MIN_FINAL_MA[name][j]
        min_ma_eta = MIN_FINAL_MA_ETA[name][j]
        min_loss = MIN_FINAL_LOSS[name][j]
        min_loss_eta = MIN_FINAL_LOSS_ETA[name][j]
        min_ma_text = f'{min_ma:.3e} @ eta={min_ma_eta:.3e}' if np.isfinite(min_ma) else 'nan'
        min_loss_text = f'{min_loss:.3e} @ eta={min_loss_eta:.3e}' if np.isfinite(min_loss) else 'nan'
        print(f'  B={B:6d}  eta_max_mean={eta_text:>9s}  eta_max_std={eta_std_text:>9s}  min_final_MA={min_ma_text}  min_final_loss={min_loss_text}')

    mask = np.isfinite(boundary) & (boundary > 0)
    if mask.sum() >= 2:
        slope, intercept = np.polyfit(np.log(B_LIST[mask]), np.log(boundary[mask]), 1)
        print(f'  fitted mean-boundary slope d log(eta) / d log(B) = {slope:.3f}')
